#### 1. Install dependencies

In [2]:
%pip install --quiet google-adk requests \
    "google-cloud-aiplatform[adk,agent_engines]" cloudpickle

#### 2. Imports/configuration

In [3]:
import getpass
from typing import Dict, Any

import os

import requests

# --- Configuration ---
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-06373caf63ce"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

# Geocoding API key is masked in the UI and not stored. It is also
# placed in an environment variable so the deployed Agent Engine
# runtime can read it (see the deployment cell's env_vars).
GOOGLE_MAPS_API_KEY = getpass.getpass("Geocoding API key: ")
os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY

# The NWS API requires a User-Agent header.
NWS_USER_AGENT = "challenge-1-weather-agent-colab (student-02-730f46eb80e1@qwiklabs.net)"
os.environ["NWS_USER_AGENT"] = NWS_USER_AGENT

Geocoding API key: ··········


#### 3. Tool: Get weather from the National Weather Service API

Takes latitude and longitude and returns the current forecast.

In [4]:
def get_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    """Retrieve the current weather forecast for a location.

    Use the U.S. National Weather Service (NWS) API. The NWS API
    resolves the latitude/longitude to a forecast grid endpoint, then
    fetches the forecast periods from that endpoint.

    Args:
        latitude: The latitude of the location in decimal degrees.
        longitude: The longitude of the location in decimal degrees.

    Returns:
        A dictionary containing the forecast. On success it has the keys:
            ``status`` (str): "success".
            ``period`` (str): The name of the forecast period (e.g. "Tonight").
            ``temperature`` (str): The temperature and unit (e.g. "72 F").
            ``forecast`` (str): A short human-readable forecast.
            ``detailed_forecast`` (str): A longer forecast description.
        On failure it returns a dictionary with keys ``status`` ("error")
        and ``error_message`` (str).
    """
    user_agent = os.environ.get("NWS_USER_AGENT", "weather-agent")
    headers = {"User-Agent": user_agent, "Accept": "application/geo+json"}

    try:
        # Step 1: Resolve the point to a forecast grid endpoint.
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        # Step 2: Fetch the forecast from the resolved endpoint.
        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]

        if not periods:
            return {
                "status": "error",
                "error_message": "No forecast periods were returned for this location.",
            }

        current = periods[0]
        return {
            "status": "success",
            "period": current["name"],
            "temperature": f"{current['temperature']} {current['temperatureUnit']}",
            "forecast": current["shortForecast"],
            "detailed_forecast": current["detailedForecast"],
        }
    except requests.exceptions.RequestException as exc:
        return {
            "status": "error",
            "error_message": (
                f"Failed to retrieve weather data: {exc}."
            ),
        }
    except (KeyError, IndexError) as exc:
        return {
            "status": "error",
            "error_message": f"Unexpected response format from NWS API: {exc}.",
        }

#### 4. Tool: Geocode a city/state using the Google Maps Geocoding API

Converts city/state into latitude and longitude.

In [5]:
def geocode_place(place: str) -> Dict[str, Any]:
    """Convert a city/state into latitude and longitude coordinates.

    Uses the Google Maps Geocoding API to resolve a place
    description (i.e. ``"Austin, TX"`` or ``"Seattle, Washington"``)
    into coordinates.

    Args:
        place: A free-form place description such as a city and state.

    Returns:
        A dictionary containing the geocoding result. On success it has
        the keys:
            ``status`` (str): "success".
            ``latitude`` (float): The latitude in decimal degrees.
            ``longitude`` (float): The longitude in decimal degrees.
            ``formatted_address`` (str): The normalized address string.
        On failure it returns a dictionary with keys ``status`` ("error")
        and ``error_message`` (str).
    """
    endpoint = "https://maps.googleapis.com/maps/api/geocode/json"
    api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
    params = {"address": place, "key": api_key}

    try:
        resp = requests.get(endpoint, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        if data.get("status") != "OK" or not data.get("results"):
            return {
                "status": "error",
                "error_message": (
                    f"Could not geocode '{place}'. API status: "
                    f"{data.get('status', 'UNKNOWN')}."
                ),
            }

        result = data["results"][0]
        location = result["geometry"]["location"]
        return {
            "status": "success",
            "latitude": location["lat"],
            "longitude": location["lng"],
            "formatted_address": result["formatted_address"],
        }
    except requests.exceptions.RequestException as exc:
        return {
            "status": "error",
            "error_message": f"Failed to reach the Geocoding API: {exc}.",
        }

#### 5. Build the agent

Register the previous two functions as tools.

In [6]:
from google.adk.agents import Agent

weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description=(
        "An agent that answers questions about the current weather for any "
        "U.S. city or state."
    ),
    instruction=(
        "You are a helpful weather assistant. When a user asks about the "
        "weather in a place (such as a city and state), follow these steps:\n"
        "1. Call the `geocode_place` tool with the place the user mentioned "
        "to obtain its latitude and longitude.\n"
        "2. Call the `get_weather` tool with that latitude and longitude to "
        "retrieve the forecast.\n"
        "3. Report the weather to the user in a friendly, concise sentence, "
        "including the temperature and a short description.\n\n"
        "If either tool returns an error status, apologize and explain the "
        "problem clearly. Remember that the National Weather Service only "
        "covers locations within the United States, so if a user asks about "
        "a place outside the U.S., let them know you can only provide U.S. "
        "weather. If the user does not specify a state, ask for clarification "
        "before geocoding."
        "4. Infer the state if you can deduce fairly accurately. \n"
        "5. If the user provides a city name that's also a state name, "
        "assume it's a city and deduce which state it's in if possible."
    ),
    tools=[geocode_place, get_weather],
)

#### 6. Deploy to Agent Platform

Deploy the weather agent onto the Agent Engine runtime.

##### 6a. Initialize Agent Engine

Point the Vertex AI SDK to the project, location, and the Cloud Storage staging bucket/path used by Agent Engine.

In [9]:
import vertexai
from vertexai import agent_engines
from vertexai.preview import reasoning_engines

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
LOCATION = os.environ["GOOGLE_CLOUD_LOCATION"]

STAGING_BUCKET = "gs://challenge-5-and-6"

# Path (prefix) for this agent's staged artifacts.
STAGING_DIR = "challenge-5"

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print(f"Vertex AI initialized for project '{PROJECT_ID}' "
      f"in '{LOCATION}' with staging bucket '{STAGING_BUCKET}/{STAGING_DIR}'.")

Vertex AI initialized for project 'qwiklabs-gcp-01-06373caf63ce' in 'us-central1' with staging bucket 'gs://challenge-5-and-6/challenge-5'.


##### 6b. Deploy the agent

Wrap the existing `weather_agent` in an `AdkApp` and deploy it to the Agent Engine runtime. The Geocoding API key is passed through as an environment variable so the deployed agent can call the Geocoding API.

In [10]:
# Wrap the ADK agent so it can run on the Agent Engine runtime.
app = reasoning_engines.AdkApp(
    agent=weather_agent,
    enable_tracing=True,
)

# Deploy to the Agent Engine runtime in the Google Cloud console.
remote_agent = agent_engines.create(
    agent_engine=app,
    display_name="weather_agent",
    description=(
        "An agent that answers questions about the current weather "
        "for any U.S. city or state."
    ),
    requirements=[
        "google-adk",
        "google-cloud-aiplatform[adk,agent_engines]",
        "requests",
        "cloudpickle",
    ],
    env_vars={
        "GOOGLE_MAPS_API_KEY": GOOGLE_MAPS_API_KEY,
        "NWS_USER_AGENT": NWS_USER_AGENT,
    },
    gcs_dir_name=STAGING_DIR,
)

print("Deployed Agent Engine resource name:")
print(remote_agent.resource_name)

INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'pydantic': '2.13.4', 'google-cloud-aiplatform': '1.162.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.13.4'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-adk', 'google-cloud-aiplatform[adk,agent_engines]', 'requests', 'cloudpickle', 'pydantic==2.13.4']
INFO:vertexai.agent_engines:Using bucket challenge-5-and-6
INFO:vertexai.agent_engines:Wrote to gs://challenge-5-and-6/challenge-5/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://challenge-5-and-6/challenge-5/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://challenge-5-and-6/challenge-5/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/571256695643/locations/us-central1/reasoningEngines/90970002176721

Deployed Agent Engine resource name:
projects/571256695643/locations/us-central1/reasoningEngines/9097000217672155136


##### 6c. Query the deployed agent

Create a session on the remote Agent Engine runtime and stream a query to confirm it works.

In [13]:
# Create a session on the remotely deployed agent.
remote_session = remote_agent.create_session(user_id="user_1")

def _reply(events):
    """Extract the agent's final text from streamed events."""
    text = ""
    for e in events:
        for p in (e.get("content", {}) or {}).get("parts", []):
            if p.get("text"):
                text = p["text"]
    return text.strip()

query = "What's the weather in Dallas, TX?"
events = list(remote_agent.stream_query(
    user_id="user_1", session_id=remote_session["id"], message=query))

print("=" * 60)
print(f"USER:  {query}")
print(f"AGENT: {_reply(events)}")
print("=" * 60)

USER:  What's the weather in Dallas, TX?
AGENT: The weather in Dallas, TX today is Sunny with a high of 101 F.


#### 7. Unit tests

Quick checks for tools and deployed agent.

In [14]:
import unittest

class TestWeatherAgent(unittest.TestCase):
    def test_geocode_success(self):
        r = geocode_place("Dallas, TX")
        self.assertEqual(r["status"], "success")
        self.assertIn("latitude", r)

    def test_geocode_failure(self):
        r = geocode_place("asdfghjkl-nowhere-000")
        self.assertEqual(r["status"], "error")

    def test_weather_success(self):
        r = get_weather(32.7767, -96.7970)
        self.assertEqual(r["status"], "success")
        self.assertIn("temperature", r)

    def test_weather_outside_us(self):
        r = get_weather(51.5074, -0.1278)
        self.assertEqual(r["status"], "error")

    def test_remote_agent(self):
        query = "What's the weather in Austin, TX?"
        s = remote_agent.create_session(user_id="test")
        events = list(remote_agent.stream_query(
            user_id="test", session_id=s["id"], message=query))
        reply = _reply(events)
        print("\n" + "-" * 60)
        print(f"USER:  {query}")
        print(f"AGENT: {reply}")
        print("-" * 60)
        self.assertTrue(events)
        self.assertTrue(reply)

unittest.main(argv=[""], exit=False, verbosity=2)

test_geocode_failure (__main__.TestWeatherAgent.test_geocode_failure) ... ok
test_geocode_success (__main__.TestWeatherAgent.test_geocode_success) ... ok
test_remote_agent (__main__.TestWeatherAgent.test_remote_agent) ... ok
test_weather_outside_us (__main__.TestWeatherAgent.test_weather_outside_us) ... 


------------------------------------------------------------
USER:  What's the weather in Austin, TX?
AGENT: The weather in Austin, TX today is Sunny with a temperature of 99 F.
------------------------------------------------------------


ok
test_weather_success (__main__.TestWeatherAgent.test_weather_success) ... ok

----------------------------------------------------------------------
Ran 5 tests in 7.727s

OK
